In [21]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

import sys
sys.path.append('..') 

from src.kitti_utils import parse_timestamps, get_frame_interval, load_oxts_velocity
from src.optical_flow import compute_sparse_flow
from src.geometry import calculate_tti_from_points

# Paths to dataset
BASE_PATH = "D:/reyci/Politecnico di Milano/2025-2/Image Analysis and CV/Project/Code/IACV-Depth-Estimation-from-Temporal-Stereo-in-Monocular-Driving-Sequences-/data/raw/2011_09_26_drive_0001_extract"
IMG_PATH = os.path.join(BASE_PATH, "image_02/data")
OXTS_PATH = os.path.join(BASE_PATH, "oxts/data")
TIME_PATH = os.path.join(BASE_PATH, "image_02/timestamps.txt")

In [22]:
print(os.listdir(BASE_PATH))

['image_00', 'image_01', 'image_02', 'image_03', 'oxts', 'velodyne_points']


In [23]:
# Define the frames to compare
idx_t = 5
idx_t1 = 6

# Load images
img_t = cv2.imread(os.path.join(IMG_PATH, f"{idx_t:010d}.png"))
img_t1 = cv2.imread(os.path.join(IMG_PATH, f"{idx_t1:010d}.png"))

# Load metadata
timestamps = parse_timestamps(TIME_PATH)
dt = get_frame_interval(timestamps, idx_t)
vf = load_oxts_velocity(OXTS_PATH, idx_t)

print(f"Delta t: {dt:.4f} s | Forward velocity vf: {vf:.2f} m/s")

Delta t: 0.1031 s | Forward velocity vf: 13.30 m/s


In [25]:
# Approximation of the FOE (typical optical center for KITTI image_02)
foe = np.array([607.19, 185.21])

# 1. Compute optical flow (moving points)
p0, p1 = compute_sparse_flow(img_t, img_t1)

# 2. Compute TTI (in seconds)
ttis = calculate_tti_from_points(p0, p1, foe, dt)

# 3. Estimate depth (Z = vf * tau)
depths_estimated = vf * ttis

AttributeError: module 'cv2' has no attribute 'calcOpticalFlowPyramidLK'

In [ ]:
plt.figure(figsize=(15, 10))
plt.imshow(cv2.cvtColor(img_t, cv2.COLOR_BGR2RGB))

for i, (old, new) in enumerate(zip(p0, p1)):
    a, b = old.ravel()
    c, d = new.ravel()

    # Draw motion vector (optical flow)
    plt.arrow(a, b, c - a, d - b, color='yellow', head_width=3)

    # Show text only for points with reasonable TTI (e.g., < 10s)
    if 0 < ttis[i] < 10:
        plt.text(
            a, b,
            f"{ttis[i]:.1f}s\n{depths_estimated[i]:.1f}m",
            color='lime',
            fontsize=8,
            fontweight='bold'
        )

plt.plot(foe[0], foe[1], 'ro', markersize=10, label='FOE (Assumed)')
plt.title(f"TTI Estimation and Depth Reconstruction (vf={vf:.1f} m/s)")
plt.legend()
plt.show()